# Pre-processing

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader


pdf_path = "data/Indias_Ancient_history_full.pdf"
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()
print(f"Pages loaded: {len(documents)}")


/var/folders/1f/dc6nbcvj0yzdh4mpnz5f_3l00000gn/T/ipykernel_42854/2781586893.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
/Users/dipyaman/Documents/codes/genAI/RAG_practice/rag/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pages loaded: 422


In [2]:
# Load opeanai api key from environment variable
import os
from dotenv import load_dotenv

load_dotenv()

open_api_key = os.environ.get("OPENAI_API_KEY")

langsmith_api_key = os.environ.get("LANGSMITH_API_KEY")
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "rag_eval_ancient"

In [3]:
full_text = " ".join([doc.page_content for doc in documents])
full_text_shorten  = " ".join([doc.page_content for doc in documents[14:]]) # Skip the first 13 pages as they are not relevant to the chapters

In [4]:
import re
pattern = r"(\d+)\.\s*\n([^\n]+)"
matches = list(re.finditer(pattern, full_text))
matches = matches[:33] 


In [5]:
chapters = []

for m in matches:  # Skip the first 33 matches
    # print("Chapter:", m.group(1))
    # print("Title:", m.group(2))
    # print("-" * 50)
    chapters.append(m.group(2))  # Append the chapter title to the list

In [6]:
import re

positions = []

for chapter in chapters:

    pattern = re.escape(chapter)
    match = re.search(pattern, full_text_shorten.replace("\n"," "), re.IGNORECASE)

    if match:
        positions.append((chapter, match.start()))

In [7]:
chapters_text = []

for i in range(0,len(positions)):

    chapter_name, start = positions[i]

    if i < len(positions) - 1:
        end = positions[i + 1][1]
    else:
        end = len(full_text_shorten)

    text = full_text_shorten[start:end].strip()

    chapters_text.append({
        "chapter": chapter_name,
        "text": text
    })

In [9]:
# Create documents
from langchain_core.documents import Document

docs = []

for ch in chapters_text:
    docs.append(Document(page_content=ch['text'], metadata={"chapter": ch['chapter']}))

chapter_dict = {item: idx for idx, item in enumerate(chapters)}

# RAG Creation

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)

from collections import defaultdict

chapter_counter = defaultdict(int)


for i, chunk in enumerate(chunks):
    chapter = chunk.metadata['chapter']
    chapter_counter[chapter] += 1
    chapter_num = chapter_dict[chapter] + 1

    chunk.metadata['chapter_chunk'] = chapter_counter[chapter]
    chunk.metadata['chunk_label'] = f"{chapter}_{chapter_counter[chapter]}"
    chunk.metadata['chunk_level'] = f"Chapter{chapter_num}_{chapter_counter[chapter]}"


print(len(chunks))

1200


### Create vectorstorage

In [14]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)
vectorstore = FAISS.from_documents(chunks, embeddings)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5582.68it/s]


In [16]:

from langsmith import traceable
retriever = vectorstore.as_retriever(
search_type="similarity",
search_kwargs={"k": 5}
)


@traceable
def generate_answer(query):
        # results = vectorstore.similarity_search(query, k=5)

        results = retriever.invoke(query)

        context = "\n".join([r.page_content for r in results])

        prompt = f"""
        You are a helpful teacher of Indian history. Answer the query ONLY based on provided context.
        Also put next line after every 100 characters in your answer. 
        If the answer is not present in the context, say "I don't know".
        Keep the output concise and suitable for a academic scholar in history.
        Context:{context}
        Query: {query}
        Answer:
        """

        from openai import OpenAI
        from langchain_openai import ChatOpenAI

        # client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
        llm = ChatOpenAI(model='gpt-4o-mini')
        response = llm.invoke(input=prompt,temperature=0.5)
        # response = client.responses.create(model='gpt-4o-mini',
                # input = prompt,
                # temperature=0.4
                # # max_output_tokens=500,
                # )
        
        return response.content




In [18]:
query = "What is the largest Buddhist temple in the world?"
answer = generate_answer(query)
print("="*100)
print(answer)
print("="*100)


The largest Buddhist temple in the world is Borobudur, located in Java. It was constructed in the eighth century and features 436 images of the Buddha illustrating his life.


# Evaluation

In [19]:
# Read the golden answer from the file
import pandas as pd
golden_df = pd.read_excel("data/golden_data_ancient.xlsx")
# golden_df =  #.set_index('Question')
golden_df['Gold_Chunk'] = golden_df['Gold_Chunk'].str.split(',')  # Strip comma from the golden chunk
display(golden_df.head())

,ID,Question,Ground Truth Chunk,Reference Answer,Gold_Chunk
0,1,Why is it important to study the history of An...,Chapter_1,The study of ancient Indian history is importa...,[Chapter1_1]
1,2,What was the role of Sir William Jones?,Chapter_2,Sir William Jones was a civil servant of East ...,"[Chapter2_1, Chapter2_2]"
2,3,How did the Rationalist approach the History?,Chapter_2,A section of the Indian scholars tries to refo...,"[Chapter2_7, Chapter2_8, Chapter2_17]"
3,4,Who was Ramakrishna Gopal Bhandarkar and what ...,Chapter_2,Ramakrishna Gopal Bhandarkar was a Indian Hist...,"[Chapter2_8, Chapter2_9]"
4,5,How are prehistoric sites are different from h...,Chapter_3,There are several difference between prehistor...,[Chapter3_1]
